In [2]:
import pandas as pd, numpy as np
from scipy.spatial.distance import squareform, pdist, cdist
from sklearn.neighbors import kneighbors_graph, NearestNeighbors
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import shortest_path

In [106]:
def diffusion_map(data, eps, alpha=1, k=4):
    Dsq = squareform(pdist(data)**2)
    Wm = np.exp(-Dsq/eps); q = Wm.sum(1)
    Wa = Wm/np.outer(q**alpha, q**alpha)
    da = Wa.sum(1); Dis = 1.0/np.sqrt(da)
    S = Dis[:, None]*Wa*Dis[None, :]                
    w, v = np.linalg.eigh(S)
    idx = np.argsort(w)[::-1]; w, v = w[idx], v[:, idx]
    phi = Dis[:, None]*v                             
    Psi = phi[:, 1:k+1]*w[1:k+1]                     
    return {"evals": w, "Psi": Psi, "phi": phi, "W": Wm, "degrees": q}

def reconstruct_path(pred, start, end):
    path = [end]; current = end
    while current != start:
        current = pred[start, current]
        if current == -9999: return None
        path.append(current)
    return path[::-1]

def graph_degree_density(Psi, h):
    Dsq = squareform(pdist(Psi)**2)
    rho = np.exp(-Dsq/h).sum(1)
    scale = rho.mean(); rho /= scale
    rho = np.maximum(rho, 1e-6)
    return rho, -np.log(rho), scale

def density_and_bandwidth(Psi, multiplier=0.1):
    N = Psi.shape[0]
    median_Dsq = np.median(squareform(pdist(Psi)**2)[np.triu_indices(N,1)])
    h = multiplier * median_Dsq
    return h, median_Dsq, graph_degree_density(Psi, h)

def density_aware_cost(A_dist_sym, V, beta):
    rows, cols = A_dist_sym.nonzero()
    base = np.asarray(A_dist_sym[rows, cols]).ravel()
    costs = base*np.exp(beta*(V[rows]+V[cols])/2)
    return csr_matrix((costs, (rows, cols)), shape=A_dist_sym.shape)

def graph_path(A_dist_sym, beta, V, start, end):
    A = density_aware_cost(A_dist_sym, V, beta)
    _, pred = shortest_path(A, directed=False, return_predecessors=True)
    return reconstruct_path(pred, start, end)

def linear_path(Psi, start_idx, end_idx, n_grid=10):
    return np.linspace(Psi[start_idx], Psi[end_idx], num=n_grid)

def latent_density_at_points(gamma, Psi, h, scale):
    Dsq_query = cdist(gamma, Psi, metric="sqeuclidean")
    rho_query = np.exp(-Dsq_query / h).sum(axis=1)
    rho_query /= scale
    rho_query_floor = np.maximum(rho_query, 1e-6)
    V_query = -np.log(rho_query_floor)
    return rho_query_floor, V_query

def local_neighbourhood_lifting(Z, Psi, gamma, start_idx, end_idx, m=27, tau=0.2):
    nn = NearestNeighbors(n_neighbors=m); nn.fit(Psi)
    distances, indices = nn.kneighbors(gamma)
    a = np.exp(-distances**2/tau) / np.sum(np.exp(-distances**2 / tau), axis=1, keepdims=True)
    points = Z[indices, :]; z_hat = np.sum(a[:, :, None] * points, axis=1)
    z_hat[0], z_hat[-1] = Z[start_idx], Z[end_idx]
    return z_hat

def lift_validation(z_hat, Z):
    nn = NearestNeighbors(n_neighbors=1); nn.fit(Z)
    dNN, _ = nn.kneighbors(z_hat)
    return dNN

In [111]:
K_GRAPH = 15
BETA = 0.75

In [113]:
df = pd.read_parquet("./datasets/joint_df_quantile.parquet")
dates = df.index; variables = df.columns[:-1]
Z = df.to_numpy()[:, :-1]; N = Z.shape[0]

diff = diffusion_map(Z, eps=3, k=3); Psi = diff["Psi"]

endpoint_pairs = {"2006 benign -> 2008 GFC": ("2006-03-01", "2008-10-01"), "2019 benign -> 2020 COVID": ("2019-07-01", "2020-04-01"), 
                  "2019 benign -> 2021 Fiscal Tightening": ("2019-04-01", "2022-04-01"), "1977 benign -> 1982 Recession Trough": ("1977-01-01", "1982-07-01")}
endpoints = {name: (dates.get_loc(pair[0]), dates.get_loc(pair[1])) for name, pair in endpoint_pairs.items()}

A_dist = kneighbors_graph(Psi, n_neighbors=K_GRAPH, mode="distance", include_self=False)
A_dist_sym = A_dist.maximum(A_dist.T)

h_dens, med_Dsq, (rho, V, scale) = density_and_bandwidth(Psi)
TAU = med_Dsq

In [ ]:
def latent_path_length(coords):
    diffs = np.diff(coords, axis=0)
    return np.linalg.norm(diffs, axis=1).sum()


def path_smoothness(coords):
    if len(coords) < 3:
        return 0.0

    second_diff = coords[2:] - 2 * coords[1:-1] + coords[:-2]
    return np.sum(np.linalg.norm(second_diff, axis=1) ** 2)


def density_weighted_cost(coords, V_path, beta):
    diffs = np.diff(coords, axis=0)
    edge_lengths = np.linalg.norm(diffs, axis=1)

    V_mid = 0.5 * (V_path[:-1] + V_path[1:])
    return np.sum(edge_lengths * np.exp(beta * V_mid))


def ambient_nn_distances(Z_path, Z_cloud):
    nn = NearestNeighbors(n_neighbors=1)
    nn.fit(Z_cloud)
    dNN, _ = nn.kneighbors(Z_path)
    return dNN.ravel()

def interior_values(x):
    x = np.asarray(x)
    if len(x) > 2:
        return x[1:-1]
    return x

In [ ]:
path_records = []

for event, (start_idx, end_idx) in endpoints.items():
    gamma = linear_path(Psi, start_idx, end_idx, n_grid=40)
    rho_lin, V_lin = latent_density_at_points(gamma, Psi, h_dens, scale)
    Z_lin = local_neighbourhood_lifting(Z, Psi, gamma, start_idx, end_idx, m=27, tau=TAU)
    path_records.append({"event": event, "path_type": "linear", "coords": gamma, "density": rho_lin, "V": V_lin, "Z_lift": Z_lin})

    idx_graph = graph_path(A_dist_sym, beta=0.0, V=V, start=start_idx, end=end_idx)
    path_records.append({"event": event, "path_type": "ordinary graph", "coords": Psi[idx_graph],
                         "density": rho[idx_graph], "V": V[idx_graph],"Z_lift": Z[idx_graph], "node_indices": idx_graph})

    idx_aware = graph_path(A_dist_sym, beta=BETA, V=V, start=start_idx, end=end_idx)
    path_records.append({"event": event,"path_type": "density-aware graph", "coords": Psi[idx_aware],
                         "density": rho[idx_aware], "V": V[idx_aware], "Z_lift": Z[idx_aware], "node_indices": idx_aware})

In [ ]:
rows = []

for rec in path_records:
    coords = rec["coords"]
    density = rec["density"]
    V_path = rec["V"]
    Z_lift = rec["Z_lift"]

    dNN = ambient_nn_distances(Z_lift, Z)
    density_int = interior_values(density)

    rows.append({
        "event": rec["event"],
        "path_type": rec["path_type"],
        "interior_minimum_density": density_int.min(),
        "interior_mean_density": density_int.mean(),
        "latent_path_length": latent_path_length(coords),
        "density_weighted_cost": density_weighted_cost(coords, V_path, beta=BETA),
        "smoothness": path_smoothness(coords),
        "max_ambient_NN_distance": dNN.max(),
        "mean_ambient_NN_distance": dNN.mean(),
    })

metrics_df = pd.DataFrame(rows)

In [119]:
for event in metrics_df["event"].unique():
    event_table = (
        metrics_df[metrics_df["event"] == event]
        .set_index("path_type")
        .drop(columns="event")
    )
    print(event)
    display(event_table.round(4))

2006 benign -> 2008 GFC


,interior_minimum_density,interior_mean_density,latent_path_length,density_weighted_cost,smoothness,max_ambient_NN_distance,mean_ambient_NN_distance
path_type,,,,,,,
linear,0.0010,0.3546,1.0535,37.2737,0.0000,1.2766,1.0039
ordinary graph,0.1706,1.1336,1.0802,5.6283,0.3846,0.0000,0.0000
density-aware graph,0.3120,1.1213,1.0921,4.6787,0.4578,0.0000,0.0000


2019 benign -> 2020 COVID


,interior_minimum_density,interior_mean_density,latent_path_length,density_weighted_cost,smoothness,max_ambient_NN_distance,mean_ambient_NN_distance
path_type,,,,,,,
linear,0.0547,0.5079,0.5500,1.8478,0.0000,1.3133,1.1017
ordinary graph,0.1861,0.6669,0.5773,1.8074,0.0236,0.0000,0.0000
density-aware graph,0.2406,0.9343,0.5904,1.7349,0.0343,0.0000,0.0000


2019 benign -> 2021 Fiscal Tightening


,interior_minimum_density,interior_mean_density,latent_path_length,density_weighted_cost,smoothness,max_ambient_NN_distance,mean_ambient_NN_distance
path_type,,,,,,,
linear,0.4402,1.5913,0.3452,0.2843,0.0000,1.2246,1.1119
ordinary graph,0.8514,1.6905,0.3765,0.3345,0.0063,0.0000,0.0000
density-aware graph,0.9182,1.7466,0.3819,0.3137,0.0056,0.0000,0.0000


1977 benign -> 1982 Recession Trough


,interior_minimum_density,interior_mean_density,latent_path_length,density_weighted_cost,smoothness,max_ambient_NN_distance,mean_ambient_NN_distance
path_type,,,,,,,
linear,0.1379,0.6405,0.3366,0.6412,0.0000,1.1979,1.0573
ordinary graph,0.3444,0.9372,0.3689,0.6611,0.0079,0.0000,0.0000
density-aware graph,0.3444,0.9372,0.3689,0.6611,0.0079,0.0000,0.0000
